## Assignment 1

Consider the sliding-tile puzzle pictured below. There are three black tiles, a blank space (empty location), and three yellow tiles, arranged randomly in the initial configuration given below:

![image.png](attachment:9f70ec57-62ad-4dbc-8130-280c4eed2394.png)

There are three legal moves:
1. (simple move) any tile can move into an adjacent empty location,
2. (jump move) any tile can jump over exactly one tile and land in the empty location, and
3. (jump move) any tile can jump over exactly two tiles and land in the empty location.

The goal is to move all the yellow tiles to the left of all the black tiles as follows:

![image.png](attachment:3823f002-1c68-4320-9c92-927d077cfee6.png)

Note that the final configuration is not unique as the blank space can be in 7 different locations and in each case the yellow and black tiles satisfy the goal conditions.

The objective is to solve the sliding-tile puzzle using various search strategies. That is, find a sequence of legal moves that takes a given "initial" configuration (start) to the specified "final" one (goal) state.

You will have to complete the code cells below. More explanations and comments need to be provided there.  

Note that in case a solution does **not** exist for some *start* and *goal* state pairs, your program will report it.

The first search strategy to be considered is Iterative Deepening Search (IDS).

Here are the required components for solving the sliding-tile puzzle using IDS:
1. State Representation: There will be a list of 6 tiles plus a blank space, with 2 colors for the tiles.
2. Goal Check: This function checks if the current state is the final state.
3. Legal Moves: The legal moves will need to be defined along with their costs.
4. Depth-Limited Search: This is a helper function to perform depth-limited search at each depth level.
5. Iterative Deepening Search (ID): Main function to perform the search which will keep calling the depth-limited search function until a solution has been found.

For IDS, we assume that the cost of any move is 1, including the jump moves over one or two tiles. In this way, IDS will find **the shortest path where the path length represents the total number of moves** to get to the final state.    

In [ ]:
# the code for the goal state check function and the generation of all possible successors states
# from the current state determined by the valid moves; there is also a helper function.

# Define the goal state check function
def check_goal(state):
    # The goal is to have all 'Y' tiles on the left of all 'B' tiles
    return state.find('B') >= state.rfind('Y')

# Generate all possible successors from the current state
def generate_successors(state):
    successors = []
    blank_index = state.index('_')

    # Move to adjacent positions
    if blank_index > 0:
        successors.append(swap(state, blank_index, blank_index - 1))
    if blank_index < 6:
        successors.append(swap(state, blank_index, blank_index + 1))

    # Jump over one tile
    if blank_index > 1:
        successors.append(swap(state, blank_index, blank_index - 2))
    if blank_index < 5:
        successors.append(swap(state, blank_index, blank_index + 2))

    # Jump over two tiles
    if blank_index > 2:
        successors.append(swap(state, blank_index, blank_index - 3))
    if blank_index < 4:
        successors.append(swap(state, blank_index, blank_index + 3))

    return successors

# Helper function to swap elements in the state
def swap(state, i, j):
    new_state = list(state)
    new_state[i], new_state[j] = new_state[j], new_state[i]
    return ''.join(new_state)

The following function implements depth-limited search using recursion. It will construct a path to the goal node while traversing the search tree up to the specified depth limit.

In particular, it takes in a state, a path and a depth level, and if the goal state has been found, it returns the path from the start state to the goal state.

In [ ]:
#Perform Depth-Limited Search (DLS)
def dls(state, depth_limit, path):
    if check_goal(state):
        return path
    if depth_limit == 0:
        return None

    for successor in generate_successors(state):
        if successor not in path:  # Avoid cycles
            result = dls(successor, depth_limit - 1, path + [successor])
            if result:
                return result

    return None

The following function implements Iterative Deepening Search (IDS) using depth-limited search.

In particular, the ids function iteratively increases the depth limit and calls depth-limited seach function dls until the goal state has been reached. In particular, it takes in an initial state, and if the goal state has been found at some depth level by depth-limited search, it returns the found path from the start state to the goal state.


In [ ]:
# Perform Iterative Deepening Search (IDS)
def ids(initial_state):
    depth = 0
    while True:
        result = dls(initial_state, depth, [initial_state])
        if result:
            return result
        depth += 1


Here is an example usage of the ids function:

In [ ]:
## Initial state example: 2 tiles per colour, 2 colours
initial_state = 'BYBY_YB'

print("Initial state:", initial_state)

# Solve the puzzle
solution_path = ids(initial_state)

# Print the solution path
if solution_path:
    print("Solution found!")
    for step in solution_path:
        print(step)
else:
    print("No solution found.")

Initial state: BYBY_YB
Solution found!
BYBY_YB
BY_YBYB
BYY_BYB
_YYBBYB
YY_BBYB
YYYBB_B


### Task 1

Your first task is to modify the program given above for solving a more general version of the sliding-tile puzzle: there will be <b>n > 0 tiles for each colour</b> plus a space, with any two colors for the tiles. In the example state given below, there are 4 black tiles, and 4 yellow tiles, and a blank space (empty location), arranged randomly in the initial configuration given below:

![Untitled 10.png](attachment:b554e499-5e39-4540-8f7f-a35732abd82b.png)

The goal is to move all the yellow tiles to the left of all the black tiles as follows:

![Untitled 12.png](attachment:1e286465-3a07-4b41-978c-9dcffab0147c.png)

Note that the final configuration is not unique as the blank space can be in 9 different locations and in each case the yellow and black tiles satisfy the goal conditions.

You will need to revise the state representation, the goal check, and the moves accordingly.
1. State Representation will have a list of <b>n > 0 tiles for each colour</b> plus a blank space, with any two given colors for the tiles.
2. Tile Colur Ordering: This is given in a list of the first and the second colours.
3. Goal Check: All the tiles of the first color will be to the left of all the tiles of the second color. As before, the blank space can be located anywhere.
4. Legal Moves: These are the same as before, a tile can be moved to an adjacent blank location, it can jump over one tile, or jump over two tiles.

Your revised Iterative Deepening Search program will print the sequence of states starting form the initial state to a goal state, along with the total number of nodes visited (explored),  the total number of nodes generated and the total runtime.

You need to provide the code where you see <b>pass</b> and add any other helper function / code you will need for your program.

In [1]:

import random
from collections import deque
import time

def is_goal(state, n, colors):
    """
    Goal: all tiles of colors[0] appear to the LEFT of all tiles of colors[1].
    Blank '_' can be anywhere.
    Also sanity-check the counts: n of each color, exactly one blank.
    """
    first, second = colors
    if state.count(first) != n or state.count(second) != n or state.count('_') != 1:
        return False

    seen_second = False
    for ch in state:
        if ch == '_':
            continue
        if ch == second:
            seen_second = True
        elif ch == first and seen_second:
            # found a first-color tile after second-color started → not goal
            return False
    return True

def swap(state, i, j):
    s = list(state)
    s[i], s[j] = s[j], s[i]
    return ''.join(s)

def generate_successors(state):
    """
    IDS/DLS version (no move costs needed).
    Allowed blank moves:
      -1, +1  -> adjacent
      -2, +2  -> jump over 1 tile
      -3, +3  -> jump over 2 tiles
    Returns: list[str] of next states.
    """
    L = len(state)
    i = state.index('_')
    deltas = [-1, 1, -2, 2, -3, 3]
    out, seen = [], set()
    for d in deltas:
        j = i + d
        if 0 <= j < L:
            nxt = swap(state, i, j)
            if nxt not in seen:
                seen.add(nxt)
                out.append(nxt)
    return out


In [2]:

# Implementations of Depth-Limited Search (DLS) and Iterative Deepening Search (IDS) with metrics

def dls(state, depth_limit, path, n, colors, stats):
    """
    Recursive depth-limited search.
    Updates stats:
      - nodes_visited: how many states were expanded (entered in DLS)
      - nodes_generated: how many successors were produced
      - max_depth: deepest recursion depth reached
    """
    stats['nodes_visited'] += 1
    stats['max_depth'] = max(stats['max_depth'], len(path)-1)

    if is_goal(state, n, colors):
        return path

    if depth_limit == 0:
        return None

    for succ in generate_successors(state):
        stats['nodes_generated'] += 1
        if succ not in path:  # avoid simple cycles along current path
            res = dls(succ, depth_limit-1, path + [succ], n, colors, stats)
            if res is not None:
                return res
    return None

def ids(initial_state, n, colors):
    """
    Iterative Deepening Search with cumulative metrics across deepening.
    Returns a dict with path + metrics as required by rubric.
    """
    t0 = time.time()
    total_generated = 0
    total_visited = 0
    max_depth_reached = 0
    depth = 0

    while True:
        stats = {'nodes_generated': 0, 'nodes_visited': 0, 'max_depth': 0}
        result_path = dls(initial_state, depth, [initial_state], n, colors, stats)

        total_generated += stats['nodes_generated']
        total_visited  += stats['nodes_visited']
        max_depth_reached = max(max_depth_reached, stats['max_depth'])

        if result_path is not None:
            runtime = time.time() - t0
            return {
                'path': result_path,
                'solution_depth': len(result_path) - 1,
                'nodes_generated': total_generated,
                'nodes_visited': total_visited,
                'max_depth_reached': max_depth_reached,
                'runtime_sec': runtime
            }

        depth += 1


Test your program with several initial configurations (either generated randomly, or directly assigned):

In [3]:

# Test cases - at least three

import random

# This function populates the tile list with n tiles of each color randomly
def create_random_initial_state(n, colors):
    tiles = [colors[0]] * n + [colors[1]] * n + ['_']
    random.shuffle(tiles)
    return ''.join(tiles)

# This function prints the solution path and metrics
def print_solution(solution_path, stats):
    if solution_path:
        print(f"Solution path ({len(solution_path)-1} moves):")
        for i, s in enumerate(solution_path):
            print(f"{i:2d}: {s}")
        sol_depth = stats.get('solution_depth', len(solution_path)-1)
        print("Solution depth:", sol_depth)
        print("Nodes explored:", stats.get('nodes_visited', 0))
        print("Nodes generated:", stats.get('nodes_generated', 0))
        print("Max depth reached:", stats.get('max_depth_reached', 0))
        print(f"Runtime (s): {stats.get('runtime_sec', 0.0):.4f}")
    else:
        print("No solution found.")


In [4]:

# Example with n = 4 and two colors
n = 4
colors = ['Y', 'B']  # two colours;

tests = []

# Medium difficulty: random configuration
tests.append(("Medium (random)", create_random_initial_state(n, colors)))


for label, initial_state in tests:
    print("\n==============================")
    print(f"{label}")
    print("Initial state:", initial_state)
    result = ids(initial_state, n, colors)
    if result:
        print_solution(result['path'], result)
    else:
        print("No solution found by IDS (unexpected).")



Medium (random)
Initial state: BYBBY_YBY
Solution path (10 moves):
 0: BYBBY_YBY
 1: BYB_YBYBY
 2: BYBYYB_BY
 3: BYBYYBYB_
 4: BYBYY_YBB
 5: BY_YYBYBB
 6: _YBYYBYBB
 7: YYB_YBYBB
 8: YYBYYB_BB
 9: YYBYY_BBB
10: YY_YYBBBB
Solution depth: 10
Nodes explored: 786026
Nodes generated: 1079348
Max depth reached: 10
Runtime (s): 1.8141


In [ ]:

# Example with n = 4 and two colors
n = 4
colors = ['Y', 'B']  # two colours;

tests = []

# Hard: worst colour ordering
tests.append(("Hard (worst-order)", colors[1]*n + '_' + colors[0]*n))

for label, initial_state in tests:
    print("\n==============================")
    print(f"{label}")
    print("Initial state:", initial_state)
    result = ids(initial_state, n, colors)
    if result:
        print_solution(result['path'], result)
    else:
        print("No solution found by IDS (unexpected).")


Hard (worst-order)
Initial state: BBBB_YYYY
Solution path (15 moves):
 0: BBBB_YYYY
 1: BB_BBYYYY
 2: BBYBB_YYY
 3: BBY_BBYYY
 4: BBYYBB_YY
 5: BBYYBBYY_
 6: BBYYB_YYB
 7: BBYYBYY_B
 8: BBYY_YYBB
 9: B_YYBYYBB
10: BYY_BYYBB
11: _YYBBYYBB
12: YY_BBYYBB
13: YYYBB_YBB
14: YYY_BBYBB
15: YYYYBB_BB
Solution depth: 15
Nodes explored: 193413751
Nodes generated: 271012563
Max depth reached: 15
Runtime (s): 501.2800


In [5]:
# Example with n = 4 and two colors
n = 4
colors = ['Y', 'B']  # two colours;

tests = []

# Easy: near-goal
tests.append(("Easy (near-goal)", colors[1] + colors[0]*n + '_' + colors[1]*(n-1)))



for label, initial_state in tests:
    print("\n==============================")
    print(f"{label}")
    print("Initial state:", initial_state)
    result = ids(initial_state, n, colors)
    if result:
        print_solution(result['path'], result)
    else:
        print("No solution found by IDS (unexpected).")


Easy (near-goal)
Initial state: BYYYY_BBB
Solution path (5 moves):
 0: BYYYY_BBB
 1: BYY_YYBBB
 2: _YYBYYBBB
 3: YY_BYYBBB
 4: YYYBY_BBB
 5: YYY_YBBBB
Solution depth: 5
Nodes explored: 1049
Nodes generated: 1525
Max depth reached: 5
Runtime (s): 0.0023


### Task 2

In the following you will implement **A\* algorithm** to find a solution to the sliding-tile puzzle as defined in **Task 1**. The A* algorithm uses a priority queue (the frontier) to explore states in the order of their estimated total cost (f_value), which is the sum of the actual cost to reach the state (g_value) and the heuristic estimate (h_value). You may choose **any admissible heuristic function** you think would be suitable for the sliding-tile puzzle problem.

For Informed Search methods such as A* algorithm and Beam Search descibed below, we assume that the cost of a simple move is 1, and the cost of a jump move is the number of tiles jumped over plus 1. In this way, the A* algorithm will find **the optimal path which minimizes the total cost along any given path**.

For the implementation of the priority queue, you should try using the functions from the module **heapq** (see <a href="https://docs.python.org/3/library/heapq.html"> "See Python Documentation for **heapq**" </a>).

Further details are given below:

1. The *heap* data structure is usually used to implement priority queues (PQ). Recall that in a PQ each item has a priority. The item with smallest value gets the highest priority.
2. An item in the queue is a tuple. The first is the estimated distance to the goal given by the chosen heuristic. This determines the priority. The second member of the tuple is the state.
3. You will need the function *heappop* to extract the item with highest priority (lowest distance).
4. Use the function *heappush* to put an item into the queue.

You need to provide the code where you see <b>pass</b> and add any other helper function / code you will need for your program.

In [6]:

import random
import heapq
import time

# Define the goal state check function
def is_goal(state, n, colors):
    """
    Goal: all tiles of colors[0] are to the LEFT of all tiles of colors[1];
    the blank '_' can be anywhere. Also sanity-check counts.
    """
    first, second = colors
    if state.count(first) != n or state.count(second) != n or state.count('_') != 1:
        return False
    seen_second = False
    for ch in state:
        if ch == '_':
            continue
        if ch == second:
            seen_second = True
        elif ch == first and seen_second:
            return False
    return True

# Helper function to swap elements in the state
def swap(state, i, j):
    s = list(state)
    s[i], s[j] = s[j], s[i]
    return ''.join(s)

# Generate all possible successors from the current state
def generate_successors(state):
    """
    Allowed blank moves:
      -1, +1  -> cost 1 (adjacent)
      -2, +2  -> cost 2 (jump over 1 tile)
      -3, +3  -> cost 3 (jump over 2 tiles)
    Returns: list of (next_state, move_cost)
    """
    L = len(state)
    i = state.index('_')
    deltas = [-1, 1, -2, 2, -3, 3]
    out, seen = [], set()
    for d in deltas:
        j = i + d
        if 0 <= j < L:
            nxt = swap(state, i, j)
            if nxt not in seen:
                seen.add(nxt)
                out.append((nxt, abs(d)))
    return out


In [7]:

import math
from heapq import heappush, heappop

# Heuristic function (admissible)
# Inversions = count of pairs (i<j) where colors[1] appears before colors[0]
def heuristic(state, n, colors):
    c0, c1 = colors[0], colors[1]
    pos_c0 = [i for i, ch in enumerate(state) if ch == c0]
    pos_c1 = [i for i, ch in enumerate(state) if ch == c1]
    inv = 0
    for i in pos_c1:
        for j in pos_c0:
            if i < j:
                inv += 1
    return math.ceil(inv / 2)

# Reconstruct the path from the goal state back to the initial state
def reconstruct_path(came_from, current):
    path = [current]
    while path[-1] in came_from:
        path.append(came_from[path[-1]])
    path.reverse()
    return path

def a_star_search(initial_state, n, colors):
    """
    A* search with:
      g = actual path cost (1 adjacent, 2 jump-1, 3 jump-2)
      h = ceil(inversions/2) (admissible)
      f = g + h
    Returns dict with path and full metrics for the rubric.
    """
    t0 = time.time()

    g_score = {initial_state: 0}
    came_from = {}

    # min-heap frontier
    open_heap = []
    heappush(open_heap, (heuristic(initial_state, n, colors), 0, initial_state))

    visited = set()
    nodes_expanded = 0
    nodes_generated = 0
    max_frontier = 1

    while open_heap:
        max_frontier = max(max_frontier, len(open_heap))
        f, g_val, state = heappop(open_heap)

        if state in visited:
            continue
        visited.add(state)
        nodes_expanded += 1

        if is_goal(state, n, colors):
            runtime = time.time() - t0
            path = reconstruct_path(came_from, state)
            return {
                "path": path,
                "solution_depth": len(path) - 1,
                "total_cost": g_val,
                "nodes_expanded": nodes_expanded,
                "nodes_generated": nodes_generated,
                "max_frontier": max_frontier,
                "runtime_sec": runtime,
                "heuristic": "Manhattan distance"
            }

        for succ, step_cost in generate_successors(state):
            tentative_g = g_val + step_cost
            if succ not in g_score or tentative_g < g_score[succ]:
                g_score[succ] = tentative_g
                came_from[succ] = state
                f_succ = tentative_g + heuristic(succ, n, colors)
                heappush(open_heap, (f_succ, tentative_g, succ))
                nodes_generated += 1

    return None


Test your program with several initial configurations (either generated randomly, or directly assigned):

In [8]:


import random

# Create a random initial state with n tiles of each color
def create_random_initial_state(n, colors):
    tiles = [colors[0]] * n + [colors[1]] * n + ['_']
    random.shuffle(tiles)
    return ''.join(tiles)

# This function prints the solution path and metrics
def print_solution(solution_path, stats):
    if solution_path:
        print(f"Solution path ({len(solution_path)-1} moves):")
        for i, s in enumerate(solution_path):
            print(f"{i:2d}: {s}")
        if isinstance(stats, dict):
            if 'total_cost' in stats:
                print("Total path cost:", stats['total_cost'])
            if 'solution_depth' in stats:
                print("Solution depth:", stats['solution_depth'])
            if 'nodes_expanded' in stats:
                print("Nodes expanded:", stats['nodes_expanded'])
            if 'nodes_generated' in stats:
                print("Nodes generated:", stats['nodes_generated'])
            if 'max_frontier' in stats:
                print("Max frontier size:", stats['max_frontier'])
            if 'runtime_sec' in stats:
                print(f"Runtime (s): {stats['runtime_sec']:.4f}")
            if 'heuristic' in stats:
                print("Heuristic:", stats['heuristic'])
    else:
        print("No solution found.")


In [9]:

# Example with n = 3 and two colors

import random
random.seed(42)

n = 3
colors = ['Y', 'B']

tests = []

# Medium: random configuration
tests.append(("Medium (random)", create_random_initial_state(n, colors)))

# Hard: worst colour ordering
tests.append(("Hard (worst-order)", colors[1]*n + '_' + colors[0]*n))

# Easy: near-goal
tests.append(("Easy (near-goal)", colors[0]*(n-1) + colors[1] + '_' + colors[0] + colors[1]*(n-1)))
# e.g., n=3 -> 'YYB_YBB' (one B leaks left)

for label, initial_state in tests:
    print("\n==============================")
    print(label)
    print("Initial state:", initial_state)
    res = a_star_search(initial_state, n, colors)
    if res:
        print_solution(res["path"], res)
    else:
        print("A* did not find a solution (unexpected).")



Medium (random)
Initial state: YBBY_YB
Solution path (4 moves):
 0: YBBY_YB
 1: Y_BYBYB
 2: YYB_BYB
 3: YY_BBYB
 4: YYYBB_B
Total path cost: 9
Solution depth: 4
Nodes expanded: 63
Nodes generated: 94
Max frontier size: 37
Runtime (s): 0.0005
Heuristic: Manhattan distance

Hard (worst-order)
Initial state: BBB_YYY
Solution path (10 moves):
 0: BBB_YYY
 1: BB_BYYY
 2: BBYB_YY
 3: BBYBYY_
 4: BBY_YYB
 5: _BYBYYB
 6: YB_BYYB
 7: Y_BBYYB
 8: YYBB_YB
 9: YY_BBYB
10: YYYBB_B
Total path cost: 22
Solution depth: 10
Nodes expanded: 134
Nodes generated: 136
Max frontier size: 25
Runtime (s): 0.0008
Heuristic: Manhattan distance

Easy (near-goal)
Initial state: YYB_YBB
Solution path (2 moves):
 0: YYB_YBB
 1: YY_BYBB
 2: YYYB_BB
Total path cost: 3
Solution depth: 2
Nodes expanded: 8
Nodes generated: 16
Max frontier size: 10
Runtime (s): 0.0001
Heuristic: Manhattan distance


### Task 3

Beam Search is a heuristic search algorithm that explores a graph by expanding the most promising nodes in a limited way. It's a variation of Best-First Search that uses a heuristic function, but unlike A*, it doesn't guarantee optimality or completeness. Instead of keeping all the promising nodes in a priority queue (the frontier), it only keeps a fixed number, or "beam," of the best nodes at each level of the search tree. This makes it much more memory-efficient than A* at the cost of potential sub-optimality.

In the following diagram, the beam width is 2 which means at any level, only two most promising nodes determined by the lowest heuristic f-scores will be inserted to the frontier. Black nodes represent the explored nodes, grey nodes are currently in the frontier, and white nodes in the rectangles have been generated and considered for insertion into the frontier (for which f-scores have been calculated); the rest of the white nodes will never be generated or considered.  

![image.png](attachment:276c2baf-8bb1-4b61-bb21-b5c16f4d84b7.png)

To implement Beam Search, we will need to adapt the core logic of a breadth-first search (BFS). The main difference is that instead of exploring all nodes at a given depth like in BFS, you only keep the k most promising ones, where k is the given beam width. If k is equal to the branching factor of the search tree, we will be doing BFS.

For Informed Search methods such as the Beam Search descibed above, we assume that the cost of a simple move is 1, however, the cost of a jump move is the number of times jumped over plus 1. Note that the Beam Search method is an anytime algorithm and it does not guarantee optimality nor completeness.

You need to provide the code where you see <b>pass</b> and add any other helper function / code you will need for your program.

In [10]:

import time
from math import ceil

def beam_search(initial_state, beam_width, n, colors, max_depth=300, per_parent=3):
    """
    Beam Search (anytime; not optimal/complete).
    f = g + h, where:
      - g = accumulated move cost using generate_successors (1 adjacent, 2 jump-1, 3 jump-2)
      - h = admissible heuristic (Task 2's heuristic if defined; fallback = ceil(inversions/2))
    Keeps only the top-k states at each depth (beam_width).
    Returns: dict with 'path', metrics, 'found_goal', and 'total_cost' when known.
    """

    #  small helper: heuristic
    def _fallback_heuristic(state):
        # Admissible lower bound: ceil(#color-inversions / 2)
        c0, c1 = colors[0], colors[1]
        pos_c0 = [i for i, ch in enumerate(state) if ch == c0]
        pos_c1 = [i for i, ch in enumerate(state) if ch == c1]
        inv = 0
        for i in pos_c1:
            for j in pos_c0:
                if i < j:
                    inv += 1
        return ceil(inv / 2)

    def h(state):

        try:
            return heuristic(state, n, colors)
        except Exception:
            return _fallback_heuristic(state)

    #  core data structures
    g_cost = {initial_state: 0}
    came_from = {}
    frontier = [initial_state]

    # metrics
    nodes_explored = 0
    nodes_generated = 0
    max_frontier = len(frontier)
    t0 = time.time()

    # anytime best-so-far (by lowest f)
    best_state = initial_state
    best_f = g_cost[initial_state] + h(initial_state)

    # avoid cross-level revisits
    global_seen = set([initial_state])

    depth = 0
    while frontier and depth <= max_depth:
        # Goal check across current beam
        for s in frontier:
            if is_goal(s, n, colors):
                runtime = time.time() - t0
                path = reconstruct_path(came_from, s)
                return {
                    "path": path,
                    "found_goal": True,
                    "nodes_explored": nodes_explored,
                    "nodes_generated": nodes_generated,
                    "runtime_sec": runtime,
                    "beam_width": beam_width,
                    "total_cost": g_cost[s],
                }

        # Expand all states in current beam level
        candidates = []
        for state in frontier:
            nodes_explored += 1
            g_here = g_cost[state]

            # Collect this parent's children first to preserve diversity
            child_list = []
            for succ, step_cost in generate_successors(state):
                if succ in global_seen:
                    continue
                nodes_generated += 1

                tentative_g = g_here + step_cost
                if succ not in g_cost or tentative_g < g_cost[succ]:
                    g_cost[succ] = tentative_g
                    came_from[succ] = state

                h_val = h(succ)
                f_val = tentative_g + h_val
                child_list.append((f_val, h_val, tentative_g, succ))

                # Anytime best
                if f_val < best_f:
                    best_f = f_val
                    best_state = succ

            # Keep only the top few children per parent (diversity)
            child_list.sort(key=lambda t: (t[0], t[1], t[2], t[3]))
            for item in child_list[:per_parent]:
                candidates.append(item)

        if not candidates:
            break

        # Global selection to top-k beam (prefer lower f, then lower h, then lower g, then lexicographic)
        candidates.sort(key=lambda t: (t[0], t[1], t[2], t[3]))
        frontier = [s for (_, _, _, s) in candidates[:beam_width]]

        # Mark picked frontier as seen to prevent cross-level revisits
        for s in frontier:
            global_seen.add(s)

        max_frontier = max(max_frontier, len(frontier))
        depth += 1

    # No goal found within beam/depth cap — return best-so-far (anytime behavior)
    runtime = time.time() - t0
    # Reconstruct best-so-far path if possible
    if best_state in came_from or best_state == initial_state:
        path = reconstruct_path(came_from, best_state) if best_state != initial_state else [initial_state]
    else:
        path = [initial_state]

    return {
        "path": path,
        "found_goal": False,
        "nodes_explored": nodes_explored,
        "nodes_generated": nodes_generated,
        "runtime_sec": runtime,
        "beam_width": beam_width,
        "total_cost": g_cost.get(best_state, None),
    }


In [11]:


import random
random.seed(42)

n = 4
colors = ['Y', 'B']
beam_width = 5

tests = [
    ("Easy case",          "YBY_BBYBY"),
    ("Hard case",          colors[1]*n + '_' + colors[0]*n),
    ("Medium case",        "YBBB_YBYY"),
]

for label, initial_state in tests:
    print("\n" + "="*30)
    print(label)
    print("Initial state:", initial_state)

    res = beam_search(initial_state, beam_width, n, colors, max_depth=300, per_parent=3)

    print("Solution found!" if res.get("found_goal") else "Best-so-far path (not goal):")
    for i, st in enumerate(res["path"]):
        print(f"Step {i}: {st}")

    # ---- Metrics for EVERY case ----
    if "total_cost" in res and res["total_cost"] is not None:
        print("Total path cost:", res["total_cost"])
    print("Number of states explored:", res.get("nodes_explored", 0))
    print("Number of states generated:", res.get("nodes_generated", 0))
    print("Time taken:", res.get("runtime_sec", 0.0), "seconds")
    if "beam_width" in res:
        print("Beam width:", res["beam_width"])
    if "note" in res:
        print("Note:", res["note"])



Easy case
Initial state: YBY_BBYBY
Solution found!
Step 0: YBY_BBYBY
Step 1: YBYB_BYBY
Step 2: YBYBYB_BY
Step 3: YBYBYBYB_
Step 4: YBYBY_YBB
Step 5: YBY_YBYBB
Step 6: YBYY_BYBB
Step 7: Y_YYBBYBB
Step 8: YYY_BBYBB
Step 9: YYYYBB_BB
Total path cost: 19
Number of states explored: 64
Number of states generated: 191
Time taken: 0.0008244514465332031 seconds
Beam width: 5

Hard case
Initial state: BBBB_YYYY
Solution found!
Step 0: BBBB_YYYY
Step 1: BBBBY_YYY
Step 2: BBB_YBYYY
Step 3: BBBYYB_YY
Step 4: BBBYYBYY_
Step 5: BBBYY_YYB
Step 6: BB_YYBYYB
Step 7: _BBYYBYYB
Step 8: YBB_YBYYB
Step 9: Y_BBYBYYB
Step 10: YYBB_BYYB
Step 11: YYBBB_YYB
Step 12: YYBBBYY_B
Step 13: YYBB_YYBB
Step 14: YYBBYY_BB
Step 15: YYB_YYBBB
Step 16: YYBYY_BBB
Step 17: YY_YYBBBB
Total path cost: 40
Number of states explored: 174
Number of states generated: 523
Time taken: 0.003877878189086914 seconds
Beam width: 5

Medium case
Initial state: YBBB_YBYY
Solution found!
Step 0: YBBB_YBYY
Step 1: YBB_BYBYY
Step 2: YBBYB_BYY


### Discussion

## Discussion

For the purposes of this assignment we utilized three search strategies for the generalized sliding-tile puzzle: Iterative Deepening Search (IDS), A*, and Beam Search. Each method was tested using easy, medium, and hard initial states, with IDS optimizing the number of moves and A*/Beam optimizing total weighted cost.

**IDS (unit-cost moves).** In IDS every move, including jumps, cost 1. This ensures the reported solution depth to be equal to the optimal number of moves. IDS is simple and guarantees shortest solution in terms of moves but is not suitable for large puzzles since it repeats and expands exponentially.

**A* (weighted-cost moves).** In A* the cost of a move reflects its difficulty: adjacent=1, jump-1=2, jump-2=3. With the admissible heuristic \(h=\lceil \text{inversions}/2 \rceil\), A* always finds the optimal total cost path. It was more efficient than IDS on medium and hard cases because the heuristic guided the search, but it required more memory to maintain the frontier.

**Beam Search (heuristic + width limit).** Beam Search also used weighted costs and the same heuristic as A*. By limiting the frontier to the top-k nodes, it reduced memory consumption and runtime drastically. But it is not guaranteed to be optimal or complete. With a small beam width it sometimes only returned best-so-far paths, but increasing the width (e.g., 5) solved even hard cases correctly. This indicates the efficiency/solution quality trade-off.

**Lessons learned.**
- IDS is reliable for finding the minimum solution in moves, but not scalable.
- A* computes optimal overall cost solutions with a trade-off of efficiency, provided that the heuristic is admissible.
- Beam Search gives speed and memory advantages, but at the cost of guarantees.
- Choice of algorithm has to depend on puzzle size and whether saving moves or saving cost is most crucial.
- Testing across in easy, medium, and hard states highlighted these trade-offs and suggested the necessity for both cost models and heuristic design.




## Overall Conclusion

This assignment successfully implemented and compared three search methods for the sliding-tile puzzle: IDS, A*, and Beam Search. IDS was shown to guarantee the shortest path in terms of moves under a unit-cost model, while A* guaranteed the optimal path in terms of weighted cost using an admissible heuristic. Beam Search demonstrated how limiting the frontier reduces memory and runtime requirements, while accepting potential sub-optimality.

Through systematic testing on easy, medium, and hard cases, we observed the strengths and weaknesses of each approach. IDS provided a strong baseline but was inefficient on larger problems. A* consistently produced optimal solutions with good efficiency, while Beam Search provided a practical compromise between solution quality and resource use.

Overall, the project highlighted the importance of defining cost models, designing heuristics, and selecting algorithms based on the problem requirements. The lessons learned reinforced how search strategies balance optimality, efficiency, and scalability, and why no single method is universally best for all cases.


#### Marking Guidelines

Here are the details of how marks will be assigned to the tasks in Assignment 1.

**10** marks are reserved for code clarity and style, and appropriate comments which explain what assumptions / decisions / changes have been made for various parts of your code. Add your comments in markdown cells and clearly identify them as **Comments**.

**25** marks are reserved for Task 1: Goal check & successor states for the more general sliding puzzle (5 marks), Depth-Limited Search (5 marks), Iterative-Deepening Search (10 marks), Testing - at least three cases (5 marks)

**30** marks are reserved for Task 2: Heuristic function (5 marks); A* algorithm with metrics (20 marks), Testing - at least three cases (5 marks)

**25** marks are reserved for Task 3: Beam Search algorithm with metrics (20 marks), Testing - at least three cases (5 marks)

**10** marks are reserved for discussion of lessons learnt, what worked and why. Add your comments in markdown cells.

**Note that tests should cover different puzzle sizes/difficulties (e.g., easy, medium, hard) to ensure robustness, not just trivial cases.**

#### Special Consideration and Late Submissions

Unless a Special Consideration request has been submitted and approved, a **5% penalty** (of the total possible mark) will be applied each day a written assessment is not submitted, up until the 7th day (including weekends). After the 7th day, a grade of '0' will be awarded even if the assessment is submitted. Submission time for all written assessments is set at 11:55 pm. A 1-hour grace period is provided to students who experience a technical concern.

For any late submission of time-sensitive tasks, such as scheduled tests/exams, performance assessments/presentations, and/or scheduled practical assessments/labs, students need to submit an application for Special Consideration.

**Assignment 1: YES, Standard Late Penalty applies**

Assignment 2: YES, Standard Late Penalty applies